# Application d'itinéraires de voyages

## Discovery
>L'objectif de cette analyse est de **déterminer les personas (au moins 2)** ainsi que les **problèmes** auxquels chacun d'eux fait face.  

### Import des données et pré traitement
>Le fichier source regroupe dans un fichier excel les réponses d'un *Google Forms* accessible [ici](https://forms.gle/8aPa21GGXQuHHGySA).

In [44]:
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

sns.set_theme()

df = pd.read_excel(r"Questionnaire réponses (RAW).xlsx")
df = df.rename( # changer le nom des colonnes
    mapper={
        "Dans quelle tranche d'âge vous situez-vous ?":"cat_age",
        "Quel type de voyage réalisez-vous le plus souvent ?":"cat_type_voyage",
        "Combien de personnes sont généralement impliquées dans la préparation de votre voyage ?":"cat_nb_orga",
        "Comment organisez-vous habituellement vos itinéraires de voyage ? (outils, méthodes, supports)":"cat_type_orga",
        "Quelles sont les principales difficultés que vous rencontrez lors de l’organisation de vos itinéraires ? (plusieurs réponses possibles)":"ls_pb_renc",
        "Avez-vous déjà eu l'impression de perdre du temps pendant un voyage ?":"bool_sent_perte_tps",
        "Si oui, à cause de quoi ?":"txt_sent_perte_tps_raisons",
        "Combien de temps consacrez-vous en moyenne à la planification de vos voyages avant de partir ?":"cat_tps_orga",
        "Quels critères sont les plus importants pour vous lors de la planification d’un itinéraire de voyage ?  (plusieurs réponses possibles)":"ls_crit_imp_orga",
        "Utilisez-vous actuellement des applications pour organiser vos voyages ?":"bool_usage_app",
        "Si oui, lesquelles ?":"ls_usage_app",
        "Qu'est-ce qui vous manque dans ces applications ?":"txt_usage_app_manque",
        "Que signifie pour vous un itinéraire « réussi » ?":"cat_itin_reussi",
        "Seriez-vous prêt(e) à utiliser une application pour optimiser l’ordre de vos visites ? ":"bool_new_app_envie",
        "Qu'est-ce qui vous ferait confiance dans une telle application ?":"cat_new_app_confiance",
        "Qu'est-ce qui vous ferait douter de son efficacité ?":"cat_new_app_doute"
    }, axis=1)

df.head()

,Horodateur,cat_age,cat_type_voyage,cat_nb_orga,cat_type_orga,ls_pb_renc,bool_sent_perte_tps,txt_sent_perte_tps_raisons,cat_tps_orga,ls_crit_imp_orga,bool_usage_app,ls_usage_app,txt_usage_app_manque,cat_itin_reussi,bool_new_app_envie,cat_new_app_confiance,cat_new_app_doute
0,2026-01-15 12:32:51.692,Entre 18 et 24 ans,"Voyage en groupe (famille, amis, collègues)","Quelques proches (ami(e)s, famille)","Application mobile (Google Maps, Tripsi, Wande...","Manque d’informations fiables, Optimisation de...",Oui,Mauvaise organisation,Plus de 6 heures,"Temps de transport, Nombre de lieux à visiter,...",Oui,Google Maps,NaN,Un itinéraire bien optimisé,Oui,Personnalisation,Erreurs dans les suggestions
1,2026-01-15 12:38:12.524,Entre 25 et 34 ans,Voyage en solo,Je planifie seul.e,"Application mobile (Google Maps, Tripsi, Wande...","Optimisation de l’itinéraire, Gestion des impr...",Non,NaN,Plus de 6 heures,"Distance entre les lieux, Temps de transport, ...",Oui,Trip advisor,On ne peut pas choisir olusieurs applis de pla...,un itinéraire flexible,Oui,Recommandations fiables,Erreurs dans les suggestions
2,2026-01-15 12:43:25.177,Entre 25 et 34 ans,"Voyage en duo (couple, ami)","Quelques proches (ami(e)s, famille)","Application mobile (Google Maps, Tripsi, Wande...","Manque d’informations fiables, Trouver des act...",Oui,Mauvaise organisation,Plus de 6 heures,"Distance entre les lieux, Nombre de lieux à vi...",Oui,Google Maps,NaN,un itinéraire flexible,Oui,Recommandations fiables,Manque de flexibilité
3,2026-01-15 12:53:04.188,Entre 25 et 34 ans,"Voyage en groupe (famille, amis, collègues)",Une équipe ou plusieurs personnes pour planifi...,"Tableur ou planification manuelle (Excel, etc.)",Gestion des imprévus,Oui,Attente entre activités,Plus de 6 heures,"Temps de transport, Budget",Oui,My maps,Les temps en transport public,Un itinéraire bien optimisé,Oui,Facilité d’utilisation,Erreurs dans les suggestions
4,2026-01-15 14:19:16.034,Entre 25 et 34 ans,"Voyage en duo (couple, ami)","Quelques proches (ami(e)s, famille)","Application mobile (Google Maps, Tripsi, Wande...","Optimisation de l’itinéraire, Gestion des impr...",Oui,Temps de transport trop long,4-6 heures,"Distance entre les lieux, Temps de transport, ...",Non,NaN,NaN,un itinéraire flexible,Non,Recommandations fiables,Confusion sur les options proposées


In [69]:
df.replace({
    'Voyage en groupe (famille, amis, collègues)':'groupe',
    'Voyage en solo':'solo',
    'Voyage en duo (couple, ami)':'duo',
    
    'Je planifie seul.e':'solo',
    'Quelques proches (ami(e)s, famille)':'duo/trio',
    'Une équipe ou plusieurs personnes pour planifier ensemble':'équipe',
    
    'Application mobile (Google Maps, Tripsi, WanderLog, TripIt etc.)':'app',
    'Google sheet + doc partagé': 'app',
    'Papier / Guide touristique':'papier',
    'Tableur ou planification manuelle (Excel, etc.)':'tableur',
    'Combinaison map et internet': 'pluriel',
    'Un peu des 3': 'pluriel',

    'Mauvaise organisation':'mauvaise orga',
    'Attente entre activités':'tps vide',
    'Temps de transport trop long':'tps transport',
    "Mauvaise organisation des transports publiques avec vélo. Ex : refus de monter à bord d'un bus.":'mauvaise orga',
    'Pour trouver un restau ou micro brasserie a notre goût':'tps recherche',

    'Distance entre les lieux':'distance',
    'Temps de transport': 'tps transport',
    'Nombre de lieux à visiter':'quantité',
    'Budget':'budget',
    'Temps de repos':'tps repos',
    'Horaires d’ouverture des lieux':'horaires',

    "Les temps en transport public":'info durée (transport)',
    "mutualisation activités/lieux/agenda":'orga',
    "Rien":'NaN',
    "La compatibilité vélo":'besoin précis (vélo)',
    "Appli trop spécialisée sur un domaine (itinéraire, hotel, restau...). Avoir une carte avec les points d'intérêts+ logements+pistes cyclables/transports et le temps de trajet équivalent entre les différentes villes":'besoin précis (vélo)',
    "Une option présentant tous les choix afin de prioriser et mettre d'autres choix pour plan de secours ":'backup',
    "Les transports locaux":'info transport',
    "Durée moyenne des visites\nMoyens de transport en commun disponibles et horaire pour les pays lointains":'info visite/info transport',
    "Je ne connais pas encore les autres applications pour bien planifier":'info app',
    "On ne peut pas choisir olusieurs applis de planification.":'',
    "Fiabilité des informations":'fiabilité info',
    "Lien avec des activités possibles dans une région":'info activité',
    "Retrouver facilement les trajets faits sur maps":'historique',
    "Informations, photos sur les endroits a visiter. Les choses à voir en dehors des sentiers battus. On se retrouve souvent dans la masse de touristes, pas forcément aux endroits les plus intéressants. Aussi ce qu'il manque c'est l'option de choisir le chemin le plus beau/intéressant pour aller d'un point a à un point b.":'',

    "Un itinéraire bien optimisé":'optimisé',
    "un itinéraire flexible":'flexible',
    "un itinéraire qui respecte mon budget":'budget',
    "un itinéraire sans imprévus":'0 imprévu',

    "Personnalisation":'personnalisable',
    "Recommandations fiables":'fiable',
    "Facilité d’utilisation":'intuitive',
    "Tout ce qui est mentionné":'all',

    "Manque de flexibilité":'peu flexible',
    "Erreurs dans les suggestions":'erreur suggestion',
    "Confusion sur les options proposées":'confusion',
    "gestion de la confidentialité des données et suivi de l'app":'confidentialité'
    
}, inplace=True)

df['ls_pb_renc'] = df['ls_pb_renc'].str.replace('Manque d’informations fiables','manque info fiable')\
                                    .str.replace('Optimisation de l’itinéraire','définition itinéraire')\
                                    .str.replace('Trouver des activités adaptées','activités')\
                                    .str.replace('Gestion des imprévus','imprévus')\
                                    .str.replace('Déplacement entre les différents endroits','définition itinéraire')\
                                    .str.replace('Trouver ce que tlm veut faire','activités')\
                                    .str.replace("Trouver un coût d'hébergement abordable qui combine les points d\'intérêts et des pistes cyclables nous y emmenant",'budget')\
                                    .str.replace("Trouver d'autres activités en fonction de joie de la météo",'météo')\
                                    .str.replace("Recherches fastidieuses pour les transports locaux tant qu'on n'est pas sur place (ex : sur maps on pensait que 2 villes seraient faciles à relier, mais en fait c'est une galère car pas de bus direct sur place, et les infos ne se trouvent pas sur internet)",'manque info fiable')\
                                    .str.replace('','')\
                                    .str.replace('','')\
                                    .str.replace('','')\
                                    .str.replace('','')

df

,Horodateur,cat_age,cat_type_voyage,cat_nb_orga,cat_type_orga,ls_pb_renc,bool_sent_perte_tps,txt_sent_perte_tps_raisons,cat_tps_orga,ls_crit_imp_orga,bool_usage_app,ls_usage_app,txt_usage_app_manque,cat_itin_reussi,bool_new_app_envie,cat_new_app_confiance,cat_new_app_doute,cat_pers
0,2026-01-15 12:32:51.692,Entre 18 et 24 ans,groupe,duo/trio,app,"manque info fiable, définition itinéraire",Oui,mauvaise orga,Plus de 6 heures,"Temps de transport, Nombre de lieux à visiter,...",Oui,Google Maps,NaN,optimisé,Oui,personnalisable,erreur suggestion,18-34
1,2026-01-15 12:38:12.524,Entre 25 et 34 ans,solo,solo,app,"définition itinéraire, imprévus",Non,NaN,Plus de 6 heures,"Distance entre les lieux, Temps de transport, ...",Oui,Trip advisor,,flexible,Oui,fiable,erreur suggestion,18-34
2,2026-01-15 12:43:25.177,Entre 25 et 34 ans,duo,duo/trio,app,"manque info fiable, activités",Oui,mauvaise orga,Plus de 6 heures,"Distance entre les lieux, Nombre de lieux à vi...",Oui,Google Maps,NaN,flexible,Oui,fiable,peu flexible,18-34
3,2026-01-15 12:53:04.188,Entre 25 et 34 ans,groupe,équipe,tableur,imprévus,Oui,tps vide,Plus de 6 heures,"Temps de transport, Budget",Oui,My maps,info durée (transport),optimisé,Oui,intuitive,erreur suggestion,18-34
4,2026-01-15 14:19:16.034,Entre 25 et 34 ans,duo,duo/trio,app,"définition itinéraire, imprévus",Oui,tps transport,4-6 heures,"Distance entre les lieux, Temps de transport, ...",Non,NaN,NaN,flexible,Non,fiable,confusion,18-34
5,2026-01-15 14:22:36.270,Entre 35 et 44 ans,groupe,duo/trio,tableur,définition itinéraire,Oui,tps transport,Plus de 6 heures,"Distance entre les lieux, Temps de transport, ...",Oui,Stippl,NaN,optimisé,Oui,fiable,erreur suggestion,35-54
6,2026-01-15 14:22:56.205,Entre 25 et 34 ans,groupe,duo/trio,tableur,activités,Non,NaN,Plus de 6 heures,"Distance entre les lieux, Temps de transport, ...",Non,NaN,NaN,flexible,Oui,intuitive,erreur suggestion,18-34
7,2026-01-15 14:26:20.227,Entre 25 et 34 ans,groupe,duo/trio,app,"définition itinéraire, activités",Oui,tps vide,Moins d’une heure,"Distance entre les lieux, Temps de transport, ...",Non,NaN,NaN,optimisé,Oui,fiable,erreur suggestion,18-34
8,2026-01-15 14:34:52.619,Entre 25 et 34 ans,duo,duo/trio,tableur,"activités, imprévus",Non,NaN,1-3 heures,"Nombre de lieux à visiter, Budget, Temps de repos",Non,NaN,NaN,budget,Oui,fiable,erreur suggestion,18-34
9,2026-01-15 14:42:03.512,Entre 25 et 34 ans,solo,solo,tableur,"définition itinéraire, imprévus",Non,NaN,Plus de 6 heures,"Nombre de lieux à visiter, Budget, Horaires d’...",Non,NaN,NaN,optimisé,Oui,fiable,erreur suggestion,18-34


### Définition des personas
>En partant de l'analyse des participants au sondage via les **questions démographiques** (*âge*) et de leurs réponses spécifiques, on peut définir plusieurs personas avec pour chacun les problèmes qu'ils rencontrent lors de l'organisation d'un voyage.

In [70]:
df.groupby("cat_age")["Horodateur"].count()

cat_age
64 ans ou plus         3
Entre 18 et 24 ans     2
Entre 25 et 34 ans    24
Entre 35 et 44 ans     1
Entre 45 et 54 ans     1
Entre 55 et 64 ans     2
Name: Horodateur, dtype: int64

>Vu la vaste majorité des participants dans la catégorie 25-34 on peut explorer pour voir si les réponses dessinent le contour d'un ou de plusieurs profils.
>
>Si un profil unique émerge dans cette tranche d'âge, on pourra regrouper celle des plus jeunes avec elle pour former un premier persona, et l'ensemble des réponses des participants plus âgés pour former un second.
>
>Si plusieurs profils typiques émergent, on pourra les scinder en plusieurs personas du même âge.

In [71]:
print(df.groupby(['cat_age', 'cat_type_voyage'])['Horodateur'].count())
print("\n",df.groupby(['cat_age', 'cat_nb_orga'])['Horodateur'].count())
print("\n",df.groupby(['cat_age', 'cat_type_orga'])['Horodateur'].count())

cat_age             cat_type_voyage
64 ans ou plus      duo                 2
                    groupe              1
Entre 18 et 24 ans  duo                 1
                    groupe              1
Entre 25 et 34 ans  duo                12
                    groupe              9
                    solo                3
Entre 35 et 44 ans  groupe              1
Entre 45 et 54 ans  groupe              1
Entre 55 et 64 ans  Duo et groupe       1
                    duo                 1
Name: Horodateur, dtype: int64

 cat_age             cat_nb_orga
64 ans ou plus      duo/trio        2
                    solo            1
Entre 18 et 24 ans  duo/trio        2
Entre 25 et 34 ans  duo/trio       13
                    solo            5
                    équipe          6
Entre 35 et 44 ans  duo/trio        1
Entre 45 et 54 ans  duo/trio        1
Entre 55 et 64 ans  duo/trio        1
                    solo            1
Name: Horodateur, dtype: int64

 cat_age             cat_

>Avant de définir les catégories selon les âges, commençons par analyser les réponses

In [77]:
df_25_34 = df[df['cat_age']=='Entre 25 et 34 ans']
df_55_plus = df[(df['cat_age']=='Entre 55 et 64 ans') | (df['cat_age']=='64 ans ou plus')]

In [89]:
# df_55_plus.groupby('cat_type_voyage')['Horodateur'].count()
df_55_plus[['cat_type_voyage','cat_nb_orga']]
# sns.heatmap(df_55_plus[['cat_type_voyage','cat_nb_orga']], x="day", y="total_bill")

,cat_type_voyage,cat_nb_orga
16,duo,solo
17,duo,duo/trio
18,Duo et groupe,duo/trio
19,duo,solo
31,groupe,duo/trio


>On peut définir les catégories comme telles:

In [78]:
def age_condition(x):
    if x in ["Entre 18 et 24 ans", "Entre 25 et 34 ans"]:
        return "18-34"
    elif x in ["Entre 35 et 44 ans", "Entre 45 et 54 ans"]:
        return "35-54"
    else:
        return "+55"

age_func = np.vectorize(age_condition)
df['cat_pers'] = age_func(df["cat_age"])

df[['cat_age','cat_pers']].head()
df.groupby('cat_pers')['cat_age'].count()

cat_pers
+55       5
18-34    26
35-54     2
Name: cat_age, dtype: int64